In [1]:
import pandas as pd

gwas = pd.read_csv(r"C:\tri_disease_project\data\raw\gwas_all.tsv", sep="\t", low_memory=False)

# search for metabolic syndrome / cardiometabolic multimorbidity traits - NOT used in Phase 2 discovery
validation_mask = gwas['DISEASE/TRAIT'].str.contains('metabolic syndrome|cardiometabolic multimorbidity', case=False, na=False) | \
                   gwas['MAPPED_TRAIT'].str.contains('metabolic syndrome|cardiometabolic multimorbidity', case=False, na=False)

gwas_validation = gwas.loc[validation_mask].copy()

print("Validation trait rows found:", len(gwas_validation))
print("\nUnique traits:")
print(gwas_validation['DISEASE/TRAIT'].value_counts())

Validation trait rows found: 3923

Unique traits:
DISEASE/TRAIT
Metabolic syndrome                                                                                                 2783
Metabolic syndrome cluster 4 (lipodystrophy-like endotype)                                                          366
Metabolic syndrome cluster 1 (non-descriptive endotype)                                                             215
Metabolic syndrome (inferred from medication usage records)                                                          82
Metabolic syndrome cluster 3 (obese endotype)                                                                        76
Cardiovascular-kidney-metabolic syndrome candidate subtype 2                                                         31
Cardiometabolic multimorbidity                                                                                       27
Serum bilirubin levels x Mediterranean diet adherence interaction in metabolic syndrome         

In [2]:
exclude_tangential = r'taste perception|bilirubin|fatty acid|antipsychotic-induced'

core_mask = validation_mask & ~(gwas['DISEASE/TRAIT'].str.contains(exclude_tangential, case=False, na=False))
gwas_validation_core = gwas.loc[core_mask].copy()

print("Core validation rows:", len(gwas_validation_core))
print(gwas_validation_core['DISEASE/TRAIT'].value_counts())

# extract genes the same way as Phase 2
def extract_gwas_genes(df):
    d = df[['MAPPED_GENE', 'P-VALUE']].dropna(subset=['MAPPED_GENE']).copy()
    d['gene_list'] = d['MAPPED_GENE'].str.split(r',\s*|\s+-\s+|\s+x\s+|;\s*')
    d = d.explode('gene_list')
    d['gene_list'] = d['gene_list'].str.strip()
    d = d[d['gene_list'] != '']
    d = d.rename(columns={'gene_list': 'symbol_raw', 'P-VALUE': 'pvalue'})
    gene_level = d.groupby('symbol_raw', as_index=False)['pvalue'].min()
    return gene_level

validation_genes_raw = extract_gwas_genes(gwas_validation_core)
print("\nUnique raw genes in metabolic syndrome GWAS:", len(validation_genes_raw))

# standardize to HGNC using the same mapping built in Phase 2
mapping = pd.read_csv(r"C:\tri_disease_project\data\interim\gene_symbol_mapping.csv")
lookup = mapping.dropna(subset=['hgnc_symbol']).set_index('symbol_raw')['hgnc_symbol'].to_dict()

validation_genes_raw['hgnc_symbol'] = validation_genes_raw['symbol_raw'].map(lookup)
validation_genes_std = validation_genes_raw.dropna(subset=['hgnc_symbol'])

# apply the same genome-wide significance threshold as Phase 3
validation_genes_sig = validation_genes_std[validation_genes_std['pvalue'] < 5e-8]

print("Standardized genes:", len(validation_genes_std))
print("Genome-wide significant genes (p<5e-8):", len(validation_genes_sig))

validation_gene_set = set(validation_genes_sig['hgnc_symbol'])

Core validation rows: 3752
DISEASE/TRAIT
Metabolic syndrome                                               2783
Metabolic syndrome cluster 4 (lipodystrophy-like endotype)        366
Metabolic syndrome cluster 1 (non-descriptive endotype)           215
Metabolic syndrome (inferred from medication usage records)        82
Metabolic syndrome cluster 3 (obese endotype)                      76
Cardiovascular-kidney-metabolic syndrome candidate subtype 2       31
Cardiometabolic multimorbidity                                     27
Dyslipidemia                                                       25
Metabolic syndrome cluster 2 (hypertensive endotype)               25
Cardiovascular-kidney-metabolic syndrome candidate subtype 16      17
Metabolic syndrome (bivariate traits)                              16
Psycho-cardiometabolic multimorbidity (multivariate analysis)      16
Cardiovascular-kidney-metabolic syndrome candidate subtype 22      15
Metabolic syndrome cluster 5 (hyperglycemic endot

In [3]:
from scipy.stats import hypergeom

hub_genes = pd.read_csv(r"C:\tri_disease_project\results\tables\hub_genes_final.csv")['gene_symbol'].tolist()
shared_genes = pd.read_csv(r"C:\tri_disease_project\data\processed\shared_genes_protein_coding.csv")['hgnc_symbol'].tolist()

hub_in_validation = [g for g in hub_genes if g in validation_gene_set]
shared_in_validation = [g for g in shared_genes if g in validation_gene_set]

print("Hub genes validated:", len(hub_in_validation), "/", len(hub_genes))
print(hub_in_validation)

print("\nShared genes validated:", len(shared_in_validation), "/", len(shared_genes))

# hypergeometric test: is this overlap more than expected by chance?
N = 20000  # genome background, same as Phase 3
K = len(shared_genes)               # size of your shared gene set
n = len(validation_gene_set)        # size of the independent validation gene set
overlap = len(shared_in_validation)

pval = hypergeom.sf(overlap - 1, N, K, n)
print(f"\nHypergeometric test - shared genes vs. independent metabolic syndrome GWAS:")
print(f"Overlap: {overlap} / {K} shared genes found in {n} validation genes")
print(f"p-value: {pval:.3e}")

# save validated gene lists
pd.Series(hub_in_validation, name='gene_symbol').to_csv(
    r"C:\tri_disease_project\results\tables\phase9_hub_genes_validated.csv", index=False)
pd.Series(shared_in_validation, name='gene_symbol').to_csv(
    r"C:\tri_disease_project\results\tables\phase9_shared_genes_validated.csv", index=False)

Hub genes validated: 4 / 17
['PPARG', 'APOE', 'LPL', 'BDNF']

Shared genes validated: 305 / 690

Hypergeometric test - shared genes vs. independent metabolic syndrome GWAS:
Overlap: 305 / 690 shared genes found in 1869 validation genes
p-value: 7.571e-135


In [4]:
print("=== PHASE 9 INDEPENDENT VALIDATION SUMMARY ===")
print(f"Validation source: GWAS Catalog 'Metabolic syndrome' traits (n={len(gwas_validation_core)} associations)")
print(f"Independent significant genes (p<5e-8): {len(validation_gene_set)}")
print(f"\nShared genes replicated: {len(shared_in_validation)}/{len(shared_genes)} ({100*len(shared_in_validation)/len(shared_genes):.1f}%)")
print(f"Hub genes replicated: {len(hub_in_validation)}/{len(hub_genes)} ({100*len(hub_in_validation)/len(hub_genes):.1f}%)")
print(f"Hypergeometric enrichment p-value: {pval:.2e}")
print(f"\nValidated hub genes: {', '.join(hub_in_validation)}")

=== PHASE 9 INDEPENDENT VALIDATION SUMMARY ===
Validation source: GWAS Catalog 'Metabolic syndrome' traits (n=3752 associations)
Independent significant genes (p<5e-8): 1869

Shared genes replicated: 305/690 (44.2%)
Hub genes replicated: 4/17 (23.5%)
Hypergeometric enrichment p-value: 7.57e-135

Validated hub genes: PPARG, APOE, LPL, BDNF
